# Energias renováveis no Brasil: APIs e aprendizado de máquina

**Aula guiada em Python | classificação e regressão**

Vamos obter dados públicos de duas APIs, transformá-los em tabelas e comparar **três algoritmos em cada problema**. Ao final, você fará uma análise própria no Orange.

**Objetivos**
1. Reconhecer o caminho API → JSON → DataFrame → CSV.
2. Distinguir classificação (prever uma categoria) de regressão (estimar um número).
3. Separar treino e teste e interpretar as métricas, sem confundir bom resultado com conclusão definitiva.

Execute as células na ordem. Em Google Colab, use **Arquivo → Salvar uma cópia no Drive** antes de começar. É necessário acesso à internet para consultar as APIs.

## Antes de começar: cinco palavras importantes

- **API:** endereço e regras para um programa pedir dados a outro sistema.
- **Requisição HTTP GET:** pedido enviado ao endereço da API, junto com parâmetros como recurso e período.
- **Autenticação:** identificação por chave ou token quando exigida. **As consultas públicas escolhidas aqui não exigem conta nem token.** Se uma API privada exigisse token, ele iria no cabeçalho `Authorization`; não se deve publicar a chave no notebook.
- **JSON:** formato estruturado da resposta, com campos e valores. Vamos converter listas do JSON em um **DataFrame**, a tabela do Pandas.
- **X e y:** `X` contém as características conhecidas; `y` é a resposta que queremos prever. No teste, ocultamos `y` do modelo e depois comparamos sua previsão com o valor conhecido.

| Problema | Fonte | `y` | Tipo |
|---|---|---|---|
| Tipo de empreendimento renovável | ANEEL/SIGA | Solar, Eólica ou Hidráulica | Classificação |
| Radiação solar horária em Petrolina (PE) | Open-Meteo | Radiação em W/m² | Regressão |

**Atenção:** radiação solar é um recurso físico disponível, não a geração elétrica de uma usina. A base da ANEEL reúne empreendimentos em diferentes fases; a classificação é um exercício de análise dos registros, não uma estimativa de geração efetiva.

## Bibliotecas e função de leitura da API

Usamos bibliotecas comuns no Colab. `urllib` e `json` vêm com Python; `pandas`, `scikit-learn` e `matplotlib` já estão presentes no Colab. A função a seguir monta a URL, envia o GET, verifica erros HTTP e lê a resposta JSON. Cada consulta tem um limite de tempo de 30 segundos.

In [ ]:
import json
import urllib.parse
import urllib.request
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

In [ ]:
def consultar_api(endereco, parametros, cabecalhos=None):
    url = endereco + "?" + urllib.parse.urlencode(parametros)
    pedido = urllib.request.Request(url, headers=cabecalhos or {})
    try:
        with urllib.request.urlopen(pedido, timeout=30) as resposta:
            return json.load(resposta)
    except Exception as erro:
        raise RuntimeError(f"Falha ao consultar a API: {erro}") from erro

# Parte 1 — Classificação: empreendimentos renováveis da ANEEL

O conjunto **SIGA** descreve empreendimentos de geração no Brasil. Vamos usar os códigos `UFV` (solar fotovoltaica), `EOL` (eólica), `UHE`, `PCH` e `CGH` (três tipos de usina hidráulica). Nosso alvo reúne os três últimos sob o rótulo *Hidráulica*.

Para prever a classe, usaremos somente **potência outorgada**, **latitude** e **longitude**. Excluímos nomes, código CEG, combustível e tipo da geração das entradas porque podem revelar a resposta. Mesmo assim, localização e potência podem diferenciar classes pela estrutura do cadastro: a atividade demonstra classificação, não prova que esses atributos causam o tipo de fonte.

A ANEEL disponibiliza o recurso CSV também por meio da API pública `datastore_search` do CKAN. Aqui o identificador do recurso é fixado para que a turma use o mesmo conjunto. A base pode ser atualizada.

### 1. Endereço, recurso e autenticação

A consulta a esse conjunto público usa GET **sem token**. `Authorization` só é preenchido se você, em outra atividade, receber um token de acesso; mantenha-o fora do arquivo compartilhado.

In [ ]:
ENDERECO_ANEEL = "https://dadosabertos.aneel.gov.br/api/3/action/datastore_search"
RECURSO_ANEEL = "11ec447d-698d-4ab8-977f-b424d5deee6a"
TOKEN_ANEEL = ""  # vazio para a consulta pública; nunca salve um token real aqui
cabecalhos_aneel = {"Authorization": TOKEN_ANEEL} if TOKEN_ANEEL else {}
print("Autenticação exigida para esta consulta pública? Não.")

### 2. Primeiro pedido e inspeção do JSON

Pedimos apenas dois registros, para enxergar a estrutura antes de carregar os dados da aula. Em `result`, o CKAN traz `records` (linhas) e `fields` (colunas).

In [ ]:
amostra_aneel = consultar_api(
    ENDERECO_ANEEL,
    {"resource_id": RECURSO_ANEEL, "limit": 2},
    cabecalhos_aneel
)
if not amostra_aneel.get("success"):
    raise ValueError("A API da ANEEL não retornou sucesso.")
print("Chaves principais:", list(amostra_aneel))
print("Campos encontrados:", [campo["id"] for campo in amostra_aneel["result"]["fields"]])

In [ ]:
display(pd.DataFrame(amostra_aneel["result"]["records"]).head(2))

### 3. Consultar cada tipo e montar a tabela

O parâmetro `filters` envia um filtro para a coluna `SigTipoGeracao`. Reunimos até 1.200 registros por tipo para manter a execução leve. A API pode devolver quantidades diferentes conforme o cadastro; depois equilibramos as três classes com uma amostra reproduzível. Isso descreve apenas a amostra coletada, não a distribuição real das fontes no Brasil.

In [ ]:
tipos = ["UFV", "EOL", "UHE", "PCH", "CGH"]
tabelas = []
for tipo in tipos:
    parametros = {
        "resource_id": RECURSO_ANEEL,
        "filters": json.dumps({"SigTipoGeracao": tipo}),
        "limit": 1200,
    }
    dados = consultar_api(ENDERECO_ANEEL, parametros, cabecalhos_aneel)
    if not dados.get("success"):
        raise ValueError(f"Consulta sem sucesso para {tipo}.")
    parte = pd.DataFrame(dados["result"]["records"])
    print(f"{tipo}: {len(parte)} linhas recebidas")
    tabelas.append(parte)
dados_aneel = pd.concat(tabelas, ignore_index=True)

### 4. Selecionar campos e tratar números

Os campos numéricos podem chegar como texto com vírgula decimal. A pequena função abaixo aceita `1234,56`, `1.234,56` e `1234.56`. Valores ausentes ou ilegíveis passam a `NaN` e são removidos. Coordenadas são aproximadas.

In [ ]:
def converter_numero(coluna):
    texto = coluna.astype(str).str.strip()
    com_virgula = texto.str.contains(",", regex=False)
    texto.loc[com_virgula] = (texto.loc[com_virgula]
                              .str.replace(".", "", regex=False)
                              .str.replace(",", ".", regex=False))
    return pd.to_numeric(texto, errors="coerce")

In [ ]:
colunas = ["SigTipoGeracao", "MdaPotenciaOutorgadaKw",
           "NumCoordNEmpreendimento", "NumCoordEEmpreendimento"]
df_class = dados_aneel[colunas].copy()
nomes = {"UFV": "Solar", "EOL": "Eólica",
         "UHE": "Hidráulica", "PCH": "Hidráulica", "CGH": "Hidráulica"}
df_class["fonte"] = df_class["SigTipoGeracao"].map(nomes)
for coluna in colunas[1:]:
    df_class[coluna] = converter_numero(df_class[coluna])
df_class = df_class.dropna(subset=colunas[1:] + ["fonte"])
df_class = df_class.rename(columns={
    "MdaPotenciaOutorgadaKw": "potencia_kw",
    "NumCoordNEmpreendimento": "latitude",
    "NumCoordEEmpreendimento": "longitude"})
df_class = df_class[["potencia_kw", "latitude", "longitude", "fonte"]]
display(df_class.head())

### 5. Conferir quantidades e criar amostra equilibrada

Comparamos apenas classes com pelo menos 30 exemplos válidos. Selecionamos até 300 de cada classe para que uma classe numerosa não esconda os erros nas outras. **Esta é uma escolha didática**, que altera as proporções observadas na base original.

In [ ]:
contagem = df_class["fonte"].value_counts()
print("Linhas válidas antes do equilíbrio:")
display(contagem.to_frame("registros"))
if len(contagem) != 3 or contagem.min() < 30:
    raise ValueError("Dados insuficientes em alguma classe. Confira a resposta da API e os campos.")
n_por_classe = min(300, int(contagem.min()))
df_class = (df_class.groupby("fonte", group_keys=False)
            .sample(n=n_por_classe, random_state=42)
            .reset_index(drop=True))
display(df_class["fonte"].value_counts().to_frame("amostra_da_aula"))

### 6. Exportar para o Orange

Salvaremos uma tabela limpa. No Colab, o arquivo aparecerá no painel **Arquivos** à esquerda; baixe-o para usar no Orange. O CSV contém `fonte` como coluna alvo e as três colunas numéricas como possíveis entradas.

In [ ]:
df_class.to_csv("aneel_classificacao_orange.csv", index=False, encoding="utf-8-sig")
print("Arquivo criado: aneel_classificacao_orange.csv")

### 7. Definir `X` e `y` e separar treino e teste

`stratify=y` preserva a proporção das três classes nos dois grupos. Os 20% de teste só são usados na avaliação. O `random_state` permite repetir a mesma divisão.

In [ ]:
from sklearn.model_selection import train_test_split
X_class = df_class[["potencia_kw", "latitude", "longitude"]]
y_class = df_class["fonte"]
Xc_treino, Xc_teste, yc_treino, yc_teste = train_test_split(
    X_class, y_class, test_size=0.20, stratify=y_class, random_state=42
)
print("Treino:", len(Xc_treino), "| Teste:", len(Xc_teste))

### 8. Treinar três classificadores

- **KNN:** consulta exemplos próximos. Como potência e coordenadas têm escalas diferentes, o `StandardScaler` é ajustado **somente com o treino** dentro de um `Pipeline`.
- **Árvore de decisão:** segue regras sucessivas.
- **Floresta aleatória:** combina decisões de muitas árvores.

Cada modelo aprende com as **mesmas linhas de treino**.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
modelo_knn = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))
modelo_knn.fit(Xc_treino, yc_treino)
print("KNN treinado")

In [ ]:
from sklearn.tree import DecisionTreeClassifier
modelo_arvore = DecisionTreeClassifier(max_depth=6, random_state=42)
modelo_arvore.fit(Xc_treino, yc_treino)
print("Árvore treinada")

In [ ]:
from sklearn.ensemble import RandomForestClassifier
modelo_floresta = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
modelo_floresta.fit(Xc_treino, yc_treino)
print("Floresta treinada")

### 9. Avaliar os três modelos

**Accuracy:** proporção total de acertos. **Precision:** quando o modelo prevê uma classe, quantas vezes acerta? **Recall:** dos exemplos reais daquela classe, quantos encontra? **F1:** combina precision e recall. A média `macro` atribui o mesmo peso a cada classe. A matriz de confusão mostra onde ocorreram os erros.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, ConfusionMatrixDisplay
modelos_class = {"KNN": modelo_knn, "Árvore": modelo_arvore,
                 "Floresta": modelo_floresta}
resultados_class = []
for nome, modelo in modelos_class.items():
    previsao = modelo.predict(Xc_teste)
    resultados_class.append({
        "Modelo": nome,
        "Accuracy": accuracy_score(yc_teste, previsao),
        "Precision macro": precision_score(yc_teste, previsao, average="macro", zero_division=0),
        "Recall macro": recall_score(yc_teste, previsao, average="macro", zero_division=0),
        "F1 macro": f1_score(yc_teste, previsao, average="macro", zero_division=0),
    })
tabela_class = pd.DataFrame(resultados_class)
display(tabela_class.round(3))

In [ ]:
fig, eixos = plt.subplots(1, 3, figsize=(15, 4))
for eixo, (nome, modelo) in zip(eixos, modelos_class.items()):
    ConfusionMatrixDisplay.from_estimator(modelo, Xc_teste, yc_teste,
                                          ax=eixo, cmap="Blues", colorbar=False)
    eixo.set_title(nome)
plt.tight_layout()
plt.show()

**Pare e interprete:** qual classe gerou mais confusões? A mudança de Accuracy acompanha a de F1 macro? Com classes equilibradas aqui, as métricas podem se aproximar; em dados reais desequilibrados, elas podem contar histórias distintas.

---

# Parte 2 — Regressão: radiação solar em Petrolina (PE)

Vamos consultar **dados horários históricos** do Open-Meteo para Petrolina em um período fixo, de **1º de abril a 30 de junho de 2025**. Um período fixo permite reproduzir a aula. A API pública usa GET sem cadastro, senha ou chave para este uso educacional. A série histórica é derivada de reanálises e modelos, portanto não equivale a medições de um sensor instalado na cidade.

Preveremos **radiação solar global horizontal média da hora (`shortwave_radiation`, W/m²)** usando temperatura, umidade, nebulosidade, vento e hora local. Ficaremos com horas de 7h a 17h para não deixar que a maior parte do exercício seja simplesmente identificar a noite. Mesmo nesse recorte, a hora é uma entrada muito informativa.

### 10. Preparar os parâmetros e consultar a API

Latitude e longitude identificam a região. `timezone` faz as horas corresponderem ao horário local. O parâmetro `hourly` pede cinco condições meteorológicas e o alvo. **Autenticação: nenhuma**. A chave `apikey` é reservada a produtos comerciais específicos e não entra nesta consulta pública.

In [ ]:
ENDERECO_METEO = "https://archive-api.open-meteo.com/v1/archive"
parametros_meteo = {
    "latitude": -9.39,
    "longitude": -40.50,
    "start_date": "2025-04-01",
    "end_date": "2025-06-30",
    "timezone": "America/Recife",
    "hourly": ",".join([
        "temperature_2m", "relative_humidity_2m", "cloud_cover",
        "wind_speed_10m", "shortwave_radiation"]),
}
print("Autenticação exigida nesta consulta pública? Não.")

In [ ]:
dados_meteo = consultar_api(ENDERECO_METEO, parametros_meteo)
if "hourly" not in dados_meteo:
    raise ValueError("A resposta não contém dados horários; confira os parâmetros.")
print("Campos da resposta:", list(dados_meteo))
print("Campos horários:", list(dados_meteo["hourly"]))

### 11. Criar e conhecer a tabela

No JSON, cada variável horária é uma lista na mesma ordem dos horários. O Pandas junta essas listas em colunas. Conferimos o começo da tabela e os valores ausentes.

In [ ]:
df_reg = pd.DataFrame(dados_meteo["hourly"])
df_reg = df_reg.rename(columns={
    "time": "data_hora", "temperature_2m": "temperatura_c",
    "relative_humidity_2m": "umidade_pct", "cloud_cover": "nuvens_pct",
    "wind_speed_10m": "vento_kmh", "shortwave_radiation": "radiacao_w_m2"})
df_reg["data_hora"] = pd.to_datetime(df_reg["data_hora"])
df_reg["hora"] = df_reg["data_hora"].dt.hour
display(df_reg.head())

In [ ]:
print("Linhas e colunas:", df_reg.shape)
display(df_reg.isna().sum().to_frame("valores_ausentes"))

### 12. Preparar as horas de interesse

Removemos apenas linhas sem as variáveis necessárias. A data fica no CSV para contextualização, mas **não entra em `X`**. Ela servirá para manter a ordem temporal da divisão treino/teste.

In [ ]:
entradas_reg = ["temperatura_c", "umidade_pct", "nuvens_pct",
                "vento_kmh", "hora"]
df_reg = df_reg[df_reg["hora"].between(7, 17)].copy()
df_reg = df_reg.dropna(subset=entradas_reg + ["radiacao_w_m2"])
df_reg = df_reg.sort_values("data_hora").reset_index(drop=True)
if len(df_reg) < 100:
    raise ValueError("Poucas observações válidas; confira as datas e a API.")
print("Observações diurnas:", len(df_reg))
display(df_reg.head())

### 13. Exportar para o Orange

Esse é o segundo CSV a baixar pelo painel **Arquivos** do Colab. Para regressão no Orange, defina `radiacao_w_m2` como alvo e use as cinco entradas indicadas. Deixe `data_hora` como metadado.

In [ ]:
df_reg.to_csv("meteo_regressao_orange.csv", index=False, encoding="utf-8-sig")
print("Arquivo criado: meteo_regressao_orange.csv")

### 14. Definir `X` e `y` e separar por tempo

Aqui `y` é um **número**, por isso usamos regressores. Treinamos com os primeiros 80% das horas e avaliamos com os 20% finais. Preservar a ordem evita que horas futuras apareçam no treino e horas anteriores no teste, embora valores de dias próximos ainda sejam relacionados.

In [ ]:
X_reg = df_reg[entradas_reg]
y_reg = df_reg["radiacao_w_m2"]
corte = int(len(df_reg) * 0.80)
Xr_treino, Xr_teste = X_reg.iloc[:corte], X_reg.iloc[corte:]
yr_treino, yr_teste = y_reg.iloc[:corte], y_reg.iloc[corte:]
print("Treino:", len(Xr_treino), "| Teste:", len(Xr_teste))
print("Primeiro horário do teste:", df_reg.loc[corte, "data_hora"])

### 15. Treinar três regressores

- **Regressão linear:** combina as entradas por uma regra linear.
- **KNN Regressor:** estima pela vizinhança; padronizamos as entradas com parâmetros aprendidos apenas no treino.
- **Random Forest Regressor:** combina estimativas de várias árvores.

Árvores não precisam dessa padronização; usamos as mesmas entradas e a mesma divisão para os três.

In [ ]:
from sklearn.linear_model import LinearRegression
modelo_linear = LinearRegression()
modelo_linear.fit(Xr_treino, yr_treino)
print("Regressão linear treinada")

In [ ]:
from sklearn.neighbors import KNeighborsRegressor
modelo_knn_reg = make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=5))
modelo_knn_reg.fit(Xr_treino, yr_treino)
print("KNN regressor treinado")

In [ ]:
from sklearn.ensemble import RandomForestRegressor
modelo_floresta_reg = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
modelo_floresta_reg.fit(Xr_treino, yr_treino)
print("Floresta regressora treinada")

### 16. Comparar os modelos

**MAE** é o erro absoluto médio em W/m²; menor é melhor. **MSE** é o erro quadrático médio em (W/m²)²; penaliza mais os erros grandes. **R²** compara o modelo com a previsão constante baseada na média do alvo no teste; próximo de 1 indica melhor ajuste, e pode ser negativo. Interprete as três métricas juntas.

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
modelos_reg = {"Linear": modelo_linear, "KNN": modelo_knn_reg,
               "Floresta": modelo_floresta_reg}
resultados_reg = []
for nome, modelo in modelos_reg.items():
    previsao = modelo.predict(Xr_teste)
    resultados_reg.append({
        "Modelo": nome,
        "MAE (W/m²)": mean_absolute_error(yr_teste, previsao),
        "MSE ((W/m²)²)": mean_squared_error(yr_teste, previsao),
        "R²": r2_score(yr_teste, previsao),
    })
tabela_reg = pd.DataFrame(resultados_reg)
display(tabela_reg.round(2))

### 17. Visualizar valores reais × previstos

Cada ponto representa uma hora do período de teste. Quanto mais perto da linha diagonal, menor o erro. O gráfico mostra a floresta como exemplo; a tabela anterior compara os três modelos.

In [ ]:
previsao_grafico = modelo_floresta_reg.predict(Xr_teste)
plt.figure(figsize=(6, 5))
plt.scatter(yr_teste, previsao_grafico, alpha=0.55, s=18)
limite = max(yr_teste.max(), previsao_grafico.max())
plt.plot([0, limite], [0, limite], "--", color="black", label="Previsão igual ao real")
plt.xlabel("Radiação real (W/m²)")
plt.ylabel("Radiação prevista (W/m²)")
plt.title("Floresta aleatória: horas do teste")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

## Agora é com vocês

Escolha **outra cidade brasileira** e **outro trimestre histórico** na API Open-Meteo. Sem receber código pronto:

1. Obtenha os dados e registre cidade, coordenadas, período e campos requisitados.
2. Formule uma pergunta de **regressão** sobre uma variável numérica disponível na API, dizendo quais colunas serão `X` e qual será `y`.
3. Treine e compare **três algoritmos** usando a mesma divisão treino/teste e MAE, MSE e R².
4. Explique um resultado inesperado, um possível erro ou uma limitação dos dados.
5. Exporte um CSV e analise também no Orange, seguindo o enunciado entregue à parte.

**Cuidados:** não use a coluna alvo (nem uma transformação direta dela) dentro de `X`. Registre se os dados são de modelo/reanálise ou medidos por sensor. Se escolher prever um valor futuro, ajuste as entradas e a divisão temporal para representar aquilo que estaria disponível no momento da previsão.

### Fontes e documentação

- [ANEEL — SIGA e dicionário do recurso](https://dadosabertos.aneel.gov.br/dataset/siga-sistema-de-informacoes-de-geracao-da-aneel)
- [ANEEL — recurso SIGA usado nesta aula](https://dadosabertos.aneel.gov.br/dataset/siga-sistema-de-informacoes-de-geracao-da-aneel/resource/11ec447d-698d-4ab8-977f-b424d5deee6a)
- [Open-Meteo — API histórica e unidades das variáveis](https://open-meteo.com/en/docs/historical-weather-api)